In [1]:
# import sys

# if 'google.colab' in sys.modules:
#     print("Running in Google Colab")
#     print("Version:", sys.version)

# else:
#     print("Not running in Google Colab")
#     print("Version:", sys.version)

In [2]:
!pip install gcsfs pyarrow   # for GCS parquet upload. running in terminal

In [3]:
import pandas as pd
import google.colab.auth

# Authenticate to Google Cloud
google.colab.auth.authenticate_user()

path = "gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet"
df = pd.read_parquet(path)

"""
    Channel mapping for eMotor dataset (current + temperature)

    Log/cDAQ9185-1F486B5Mod1/ai0 -> temp_A
    Log/cDAQ9185-1F486B5Mod1/ai1 -> temp_B
    Log/cDAQ9185-1F486B5Mod2/ai0 -> current_u
    Log/cDAQ9185-1F486B5Mod2/ai2 -> current_v
    Log/cDAQ9185-1F486B5Mod2/ai3 -> current_w
"""

print(df.shape)
print(df.columns)
df.head()

(1536492, 5)
Index(['Log/cDAQ9185-1F486B5Mod1/ai0', 'Log/cDAQ9185-1F486B5Mod1/ai1',
       'Log/cDAQ9185-1F486B5Mod2/ai0', 'Log/cDAQ9185-1F486B5Mod2/ai2',
       'Log/cDAQ9185-1F486B5Mod2/ai3'],
      dtype='object')


,Log/cDAQ9185-1F486B5Mod1/ai0,Log/cDAQ9185-1F486B5Mod1/ai1,Log/cDAQ9185-1F486B5Mod2/ai0,Log/cDAQ9185-1F486B5Mod2/ai2,Log/cDAQ9185-1F486B5Mod2/ai3
0,27.607992,28.217591,1.894377,0.949463,-2.165271
1,27.607992,28.217591,2.128889,0.926111,-2.309938
2,27.607992,28.217591,2.373001,0.882154,-2.526938
3,27.607992,28.217591,2.087747,1.193974,-2.444867
4,27.607992,28.217591,2.393572,1.125291,-2.718899


In [4]:
path = "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet"
df_vibration = pd.read_parquet(path)

"""
    Channel mapping for eMotor dataset (vibration)

    ch1 -> vib_x_A,
    ch2 -> vib_y_A,
    ch3 -> vib_x_B,
    ch4 -> vib_y_B
"""

print(df_vibration.shape)
print(df_vibration.columns)
df_vibration.head()

(1536000, 5)
Index(['ch1', 'ch2', 'ch3', 'ch4', 'time'], dtype='object')


,ch1,ch2,ch3,ch4,time
0,-8.947968,14.536224,-1.054167,0.969664,0.000012
1,2.321366,-2.065806,-1.840423,3.046795,0.000051
2,0.273554,2.071666,-0.860117,2.146393,0.000090
3,-7.890409,15.446308,-0.465193,1.381279,0.000129
4,-1.295177,-0.780089,0.003352,-0.305207,0.000168


### Parse torque and label from filename

In [5]:
import re

def parse_metadata(filename: str):
    """
    Ejemplo filename: 0Nm_BPFI_03.parquet
    """
    torque = int(re.search(r"(\d)Nm", filename).group(1))

    if "Normal" in filename:
        label = "Normal"
    elif "BPFI" in filename:
        label = "BPFI"
    elif "BPFO" in filename:
        label = "BPFO"
    else:
        label = None

    return torque, label

### Load and std dataframe: current + temperature

In [6]:
import pandas as pd

def load_current_temp(path):
    df = pd.read_parquet(path)

    df = df.rename(columns={
        "Log/cDAQ9185-1F486B5Mod2/ai0": "current_u",
        "Log/cDAQ9185-1F486B5Mod2/ai2": "current_v",
        "Log/cDAQ9185-1F486B5Mod2/ai3": "current_w",
        "Log/cDAQ9185-1F486B5Mod1/ai0": "temp_A",
        "Log/cDAQ9185-1F486B5Mod1/ai1": "temp_B"
    })

    return df.reset_index(drop=True)

### Load and std dataframe: vibration

In [7]:
def load_vibration(path):
    df = pd.read_parquet(path)

    df = df.rename(columns={
        "ch1": "vib_x_A",
        "ch2": "vib_y_A",
        "ch3": "vib_x_B",
        "ch4": "vib_y_B"
    })

    return df[["vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"]].reset_index(drop=True)

### Merge current/temp and vibration dataframes

In [8]:
def merge_signals(df_curr, df_vib):
    n = min(len(df_curr), len(df_vib))

    df = pd.concat(
        [df_curr.iloc[:n], df_vib.iloc[:n]],
        axis=1
    )

    return df


### Windowing

In [9]:
import numpy as np

def sliding_window(df, window_size, step_size):
    """
    df: DataFrame con señales
    window_size: nº de samples por ventana
    step_size: salto entre ventanas
    """
    windows = []

    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        windows.append(window)

    return windows

### Feature extraction (ML)
##### **mean:** promedio aritmético de todos los valores de la señal en un tiempo determinado. En una señal de corriente alterna (AC)     perfecta, la media es cero; Si no es cero, indica una componente de corriente continua (DC)

##### **std:** mide la variación los datos respecto a la media. En señales eléctricas sin componente DC, el valor de la desviación estándar es prácticamente igual al valor RMS.

##### **RMS:** valor eficaz de la corriente, es decir, el valor de una corriente continua que produciría la misma disipación de calor que la señal alterna

##### **kurtosis:** mide qué tan "puntiaguda" o "achatada" es la distribución de la señal. En análisis de vibraciones o corrientes, una curtosis alta suele indicar la presencia de picos transitorios o impactos (como un rodamiento dañado o un arco eléctrico), ya que hay valores extremos fuera de lo común.

##### **skew:** asimetría. Indica si la señal es simétrica respecto a la media. Una señal balanceada tiene asimetría cero. Si hay fallos en un rectificador, por ejemplo, la señal puede "cargarse" más hacia un lado, aumentando la asimetría.

In [10]:
# Define fast and slow signal names
FAST_SIGNALS = [
    "current_u", "current_v", "current_w",
    "vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"
]

SLOW_SIGNALS = [
    "temp_A", "temp_B"
]

In [11]:
# Feature extraction functions
from scipy.stats import kurtosis, skew

def extract_time_features(window):
    features = {}

    # Señales rápidas
    for col in FAST_SIGNALS:
        x = window[col].values
        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)
        features[f"{col}_rms"] = np.sqrt(np.mean(x**2))
        features[f"{col}_kurtosis"] = kurtosis(x)
        features[f"{col}_skew"] = skew(x)

    # Señales lentas (temperatura)
    for col in SLOW_SIGNALS:
        x = window[col].values
        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)

    return features

In [12]:
# from scipy.stats import kurtosis, skew

# def extract_time_features(window):
#     features = {}

#     for col in window.columns:
#         x = window[col].values

#         features[f"{col}_mean"] = np.mean(x)
#         features[f"{col}_std"] = np.std(x)
#         features[f"{col}_rms"] = np.sqrt(np.mean(x**2))
#         features[f"{col}_kurtosis"] = kurtosis(x)
#         features[f"{col}_skew"] = skew(x)

#     return features


### All pipeline

In [13]:
def process_file(
    curr_path,
    vib_path,
    filename,
    window_size,
    step_size
):
    df_curr = load_current_temp(curr_path)
    df_vib  = load_vibration(vib_path)

    df = merge_signals(df_curr, df_vib)

    torque, label = parse_metadata(filename)
    if label is None:
        return None

    windows = sliding_window(df, window_size, step_size)

    rows = []
    for w in windows:
        feats = extract_time_features(w)
        feats["label"] = label
        feats["torque_nm"] = torque
        feats["source_file"] = filename
        rows.append(feats)

    return pd.DataFrame(rows)


In [14]:
SAMPLING_RATE = 10000   # ejemplo (ajustable)
WINDOW_SECONDS = 1.0

WINDOW_SIZE = int(SAMPLING_RATE * WINDOW_SECONDS)
STEP_SIZE   = int(WINDOW_SIZE * 0.5)

In [15]:
df_ml = process_file("gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet",
                     "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet",
                     "0Nm_BPFI_03",
                     2_000,
                     1_000)

print(df_ml.shape)
print(df_ml.columns)
df_ml.head()

(1535, 42)
Index(['current_u_mean', 'current_u_std', 'current_u_rms',
       'current_u_kurtosis', 'current_u_skew', 'current_v_mean',
       'current_v_std', 'current_v_rms', 'current_v_kurtosis',
       'current_v_skew', 'current_w_mean', 'current_w_std', 'current_w_rms',
       'current_w_kurtosis', 'current_w_skew', 'vib_x_A_mean', 'vib_x_A_std',
       'vib_x_A_rms', 'vib_x_A_kurtosis', 'vib_x_A_skew', 'vib_y_A_mean',
       'vib_y_A_std', 'vib_y_A_rms', 'vib_y_A_kurtosis', 'vib_y_A_skew',
       'vib_x_B_mean', 'vib_x_B_std', 'vib_x_B_rms', 'vib_x_B_kurtosis',
       'vib_x_B_skew', 'vib_y_B_mean', 'vib_y_B_std', 'vib_y_B_rms',
       'vib_y_B_kurtosis', 'vib_y_B_skew', 'temp_A_mean', 'temp_A_std',
       'temp_B_mean', 'temp_B_std', 'label', 'torque_nm', 'source_file'],
      dtype='object')


,current_u_mean,current_u_std,current_u_rms,current_u_kurtosis,current_u_skew,current_v_mean,current_v_std,current_v_rms,current_v_kurtosis,current_v_skew,...,vib_y_B_rms,vib_y_B_kurtosis,vib_y_B_skew,temp_A_mean,temp_A_std,temp_B_mean,temp_B_std,label,torque_nm,source_file
0,0.051824,2.369356,2.369923,-1.534883,0.007640,-0.256863,2.293489,2.307829,-1.434533,0.103552,...,1.825614,0.489534,0.050802,27.606317,0.001068,28.207880,6.191117e-03,BPFI,0,0Nm_BPFI_03
1,0.129027,2.339341,2.342897,-1.496389,-0.076224,-0.273917,2.266691,2.283182,-1.425187,0.092111,...,1.812273,0.474041,0.049192,27.605636,0.000000,28.203933,7.105427e-15,BPFI,0,0Nm_BPFI_03
2,0.170414,2.279735,2.286095,-1.453073,-0.088774,-0.201464,2.310442,2.319209,-1.451804,0.093089,...,1.795875,0.283869,0.060849,27.605636,0.000000,28.203933,7.105427e-15,BPFI,0,0Nm_BPFI_03
3,0.183585,2.274952,2.282347,-1.451941,-0.093499,-0.093555,2.360467,2.362321,-1.510567,0.034355,...,1.824445,0.207594,0.100616,27.605636,0.000000,28.203933,7.105427e-15,BPFI,0,0Nm_BPFI_03
4,0.136873,2.332955,2.336966,-1.498397,-0.081313,0.005902,2.344801,2.344808,-1.485462,-0.057294,...,1.826338,0.222346,0.109016,27.605463,0.001145,28.204321,2.556980e-03,BPFI,0,0Nm_BPFI_03


### Feature engineering in freq: FFT features

In [16]:
def compute_fft(x, fs):
    """FFT normalizada (solo magnitud positiva)."""
    x = x - np.mean(x)
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    mag = np.abs(X)
    return freqs, mag

In [17]:
# Features espectrales por señal

def spectral_features(x, fs):
    freqs, mag = compute_fft(x, fs)

    power = mag**2
    total_energy = np.sum(power)

    # Bandas (fracciones de Nyquist)
    nyq = fs / 2
    low_band  = freqs <= nyq * 0.2
    mid_band  = (freqs > nyq * 0.2) & (freqs <= nyq * 0.5)
    high_band = freqs > nyq * 0.5

    energy_low  = np.sum(power[low_band])
    energy_mid  = np.sum(power[mid_band])
    energy_high = np.sum(power[high_band])

    # Entropía espectral
    psd_norm = power / (total_energy + 1e-12)
    spec_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))

    return {
        "spec_energy_total": total_energy,
        "spec_energy_low": energy_low,
        "spec_energy_mid": energy_mid,
        "spec_energy_high": energy_high,
        "spec_entropy": spec_entropy
    }


In [18]:
# Integrar FFT al extractor por ventana

DYNAMIC_COLS = [
    "current_u", "current_v", "current_w",
    "vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"
]

def extract_freq_features(window, fs):
    feats = {}
    for col in DYNAMIC_COLS:
        x = window[col].values
        spec = spectral_features(x, fs)
        for k, v in spec.items():
            feats[f"{col}_{k}"] = v
    return feats


In [19]:
# Unir tiempo + frecuencia en el extractor de características

def extract_features(window, fs):
    feats_time = extract_time_features(window)
    feats_freq = extract_freq_features(window, fs)
    feats_time.update(feats_freq)
    return feats_time

In [20]:
# Update pipeline para usar extractor combinado

def process_file_fft(
    curr_path,
    vib_path,
    filename,
    window_size,
    step_size,
    fs
):
    df_curr = load_current_temp(curr_path)
    df_vib  = load_vibration(vib_path)
    df = merge_signals(df_curr, df_vib)

    torque, label = parse_metadata(filename)
    if label is None:
        return None

    windows = sliding_window(df, window_size, step_size)

    rows = []
    for w in windows:
        feats = extract_features(w, fs)
        feats["label"] = label
        feats["torque_nm"] = torque
        feats["source_file"] = filename
        rows.append(feats)

    return pd.DataFrame(rows)


In [21]:
df_ml_global = process_file_fft("gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet",
                     "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet",
                     "0Nm_BPFI_03",
                     2_000,
                     1_000,
                     25600)

print(df_ml_global.shape)
print(df_ml_global.columns)
df_ml_global.head()

(1535, 77)
Index(['current_u_mean', 'current_u_std', 'current_u_rms',
       'current_u_kurtosis', 'current_u_skew', 'current_v_mean',
       'current_v_std', 'current_v_rms', 'current_v_kurtosis',
       'current_v_skew', 'current_w_mean', 'current_w_std', 'current_w_rms',
       'current_w_kurtosis', 'current_w_skew', 'vib_x_A_mean', 'vib_x_A_std',
       'vib_x_A_rms', 'vib_x_A_kurtosis', 'vib_x_A_skew', 'vib_y_A_mean',
       'vib_y_A_std', 'vib_y_A_rms', 'vib_y_A_kurtosis', 'vib_y_A_skew',
       'vib_x_B_mean', 'vib_x_B_std', 'vib_x_B_rms', 'vib_x_B_kurtosis',
       'vib_x_B_skew', 'vib_y_B_mean', 'vib_y_B_std', 'vib_y_B_rms',
       'vib_y_B_kurtosis', 'vib_y_B_skew', 'temp_A_mean', 'temp_A_std',
       'temp_B_mean', 'temp_B_std', 'current_u_spec_energy_total',
       'current_u_spec_energy_low', 'current_u_spec_energy_mid',
       'current_u_spec_energy_high', 'current_u_spec_entropy',
       'current_v_spec_energy_total', 'current_v_spec_energy_low',
       'current_v_spec_e

,current_u_mean,current_u_std,current_u_rms,current_u_kurtosis,current_u_skew,current_v_mean,current_v_std,current_v_rms,current_v_kurtosis,current_v_skew,...,vib_x_B_spec_energy_high,vib_x_B_spec_entropy,vib_y_B_spec_energy_total,vib_y_B_spec_energy_low,vib_y_B_spec_energy_mid,vib_y_B_spec_energy_high,vib_y_B_spec_entropy,label,torque_nm,source_file
0,0.051824,2.369356,2.369923,-1.534883,0.007640,-0.256863,2.293489,2.307829,-1.434533,0.103552,...,1.782923e+06,4.429156,6.665684e+06,1.894287e+06,2.095149e+06,2.676248e+06,4.549384,BPFI,0,0Nm_BPFI_03
1,0.129027,2.339341,2.342897,-1.496389,-0.076224,-0.273917,2.266691,2.283182,-1.425187,0.092111,...,1.579193e+06,4.413529,6.568637e+06,1.933064e+06,2.133243e+06,2.502331e+06,4.485526,BPFI,0,0Nm_BPFI_03
2,0.170414,2.279735,2.286095,-1.453073,-0.088774,-0.201464,2.310442,2.319209,-1.451804,0.093089,...,1.557209e+06,4.373759,6.450252e+06,1.761194e+06,2.076341e+06,2.612717e+06,4.504422,BPFI,0,0Nm_BPFI_03
3,0.183585,2.274952,2.282347,-1.451941,-0.093499,-0.093555,2.360467,2.362321,-1.510567,0.034355,...,1.737493e+06,4.481906,6.657105e+06,1.903655e+06,2.097715e+06,2.655735e+06,4.524315,BPFI,0,0Nm_BPFI_03
4,0.136873,2.332955,2.336966,-1.498397,-0.081313,0.005902,2.344801,2.344808,-1.485462,-0.057294,...,1.579556e+06,4.385036,6.670902e+06,2.056025e+06,2.145824e+06,2.469053e+06,4.497322,BPFI,0,0Nm_BPFI_03


In [22]:
df_ml_global.describe()

,current_u_mean,current_u_std,current_u_rms,current_u_kurtosis,current_u_skew,current_v_mean,current_v_std,current_v_rms,current_v_kurtosis,current_v_skew,...,vib_x_B_spec_energy_low,vib_x_B_spec_energy_mid,vib_x_B_spec_energy_high,vib_x_B_spec_entropy,vib_y_B_spec_energy_total,vib_y_B_spec_energy_low,vib_y_B_spec_energy_mid,vib_y_B_spec_energy_high,vib_y_B_spec_entropy,torque_nm
count,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,1535.000000,...,1.535000e+03,1535.000000,1.535000e+03,1535.000000,1.535000e+03,1.535000e+03,1.535000e+03,1.535000e+03,1535.000000,1535.0
mean,0.034828,2.318359,2.321712,-1.485375,0.001002,-0.089892,2.318138,2.322958,-1.462641,0.008855,...,8.612312e+06,340894.997268,1.638223e+06,4.419868,6.589665e+06,1.889174e+06,2.229392e+06,2.471099e+06,4.621920,0.0
std,0.119212,0.035695,0.033634,0.031655,0.076715,0.118994,0.036706,0.034784,0.032897,0.075692,...,5.700243e+05,30103.420772,1.495302e+05,0.092117,2.836673e+05,9.719321e+04,1.370240e+05,1.988449e+05,0.084565,0.0
min,-0.169390,2.250105,2.253938,-1.540684,-0.100754,-0.291572,2.249674,2.252051,-1.518794,-0.090858,...,7.127461e+06,270022.890248,1.223663e+06,3.946177,5.623938e+06,1.596559e+06,1.812003e+06,1.958268e+06,4.284158,0.0
25%,-0.082553,2.284073,2.289947,-1.516808,-0.084139,-0.206745,2.283373,2.290766,-1.495642,-0.074234,...,8.209896e+06,320145.447784,1.531748e+06,4.359891,6.387225e+06,1.822696e+06,2.133025e+06,2.330711e+06,4.568448,0.0
50%,0.034990,2.318788,2.322023,-1.477587,0.001424,-0.089112,2.318175,2.324557,-1.454850,0.008744,...,8.595513e+06,337056.327572,1.633624e+06,4.421571,6.592070e+06,1.887911e+06,2.222597e+06,2.461183e+06,4.622602,0.0
75%,0.152827,2.352316,2.353401,-1.454673,0.086130,0.026123,2.352676,2.355357,-1.431388,0.092778,...,8.979400e+06,355008.168098,1.734035e+06,4.484522,6.773801e+06,1.955116e+06,2.320883e+06,2.601105e+06,4.673901,0.0
max,0.240133,2.383477,2.384602,-1.442919,0.103408,0.106531,2.383822,2.385510,-1.415216,0.111467,...,1.101481e+07,532926.538512,2.277054e+06,4.727515,7.700369e+06,2.289139e+06,2.711411e+06,3.248231e+06,4.952379,0.0


In [23]:
df_ml_global.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1535 entries, 0 to 1534
Data columns (total 77 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   current_u_mean               1535 non-null   float64
 1   current_u_std                1535 non-null   float64
 2   current_u_rms                1535 non-null   float64
 3   current_u_kurtosis           1535 non-null   float64
 4   current_u_skew               1535 non-null   float64
 5   current_v_mean               1535 non-null   float64
 6   current_v_std                1535 non-null   float64
 7   current_v_rms                1535 non-null   float64
 8   current_v_kurtosis           1535 non-null   float64
 9   current_v_skew               1535 non-null   float64
 10  current_w_mean               1535 non-null   float64
 11  current_w_std                1535 non-null   float64
 12  current_w_rms                1535 non-null   float64
 13  current_w_kurtosis

### 4. Batch processing

In [24]:
# Define base paths
from pathlib import Path
import gcsfs

fs = gcsfs.GCSFileSystem()

CURR_PATH = "gs://emotor-dataset-raw/current_temp_short/"
VIBR_PATH = "gs://emotor-dataset-raw/vibration_temp_short/"

curr_files = [
    Path(f).name for f in fs.ls(CURR_PATH)
    if f.endswith(".parquet")
]

vibr_files = [
    Path(f).name for f in fs.ls(VIBR_PATH)
    if f.endswith(".parquet")
]

common_files = sorted(set(curr_files).intersection(vibr_files))
print(f"Archivos comunes: {len(common_files)}")

Archivos comunes: 40


In [25]:
# List files in GCS buckets
def is_valid_file(filename: str) -> bool:
    return any(k in filename for k in ["Normal", "BPFI", "BPFO"])


selected_files = [
    f for f in common_files
    if is_valid_file(f)
]

print(f"Archivos seleccionados: {len(selected_files)}")

Archivos seleccionados: 21


In [26]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_single_file(filename):
    curr_path = CURR_PATH + filename
    vib_path  = VIBR_PATH + filename

    try:
        df_part = process_file_fft(
            curr_path=curr_path,
            vib_path=vib_path,
            filename=filename,
            window_size=2_000,
            step_size=1_000,
            fs=25_600
        )
        return df_part

    except Exception as e:
        return f"ERROR::{filename}::{e}"

In [28]:
dfs = []
errors = []

for i, filename in enumerate(selected_files, 1):
    curr_path = CURR_PATH + filename
    vib_path  = VIBR_PATH + filename

    try:
        df_part = process_file_fft(
            curr_path=curr_path,
            vib_path=vib_path,
            filename=filename,
            window_size=2_000,
            step_size=1_000,
            fs=25_600
        )

        if df_part is not None:
            dfs.append(df_part)

        if i % 3 == 0:
            print(f"Procesados {i}/{len(selected_files)} archivos")

    except Exception as e:
        errors.append(f"ERROR::{filename}::{e}")
        print(errors[-1])


Procesados 3/21 archivos
Procesados 6/21 archivos
Procesados 9/21 archivos
Procesados 12/21 archivos
Procesados 15/21 archivos
Procesados 18/21 archivos
Procesados 21/21 archivos


In [39]:
df_ml_global.shape

(41451, 77)

In [40]:
print(len(selected_files))
print(len(dfs))
print(selected_files)

21
21
['0Nm_BPFI_03.parquet', '0Nm_BPFI_10.parquet', '0Nm_BPFI_30.parquet', '0Nm_BPFO_03.parquet', '0Nm_BPFO_10.parquet', '0Nm_BPFO_30.parquet', '0Nm_Normal.parquet', '2Nm_BPFI_03.parquet', '2Nm_BPFI_10.parquet', '2Nm_BPFI_30.parquet', '2Nm_BPFO_03.parquet', '2Nm_BPFO_10.parquet', '2Nm_BPFO_30.parquet', '2Nm_Normal.parquet', '4Nm_BPFI_03.parquet', '4Nm_BPFI_10.parquet', '4Nm_BPFI_30.parquet', '4Nm_BPFO_03.parquet', '4Nm_BPFO_10.parquet', '4Nm_BPFO_30.parquet', '4Nm_Normal.parquet']


In [41]:
df_ml_global = pd.concat(dfs, ignore_index=True)
print(df_ml_global.shape)

(41451, 77)


In [43]:
df_ml_global.to_parquet(
    "gs://emotor-dataset-processed/df_ml_global.parquet",
    index=False
)